# Dataset Jurídico e de Segurança Pública — Aula 10
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública

Este notebook cria o banco de dados SQLite completo com dados jurídicos e de segurança pública usados em todos os Labs da Aula 10.

**Execute este notebook ANTES dos Labs 1, 2 e 3.**

---

### Estrutura do Dataset

- **`acordaos`** — Acórdãos do STF e STJ (HC, AP, RHC)
- **`ocorrencias`** — Boletins de ocorrência policial
- **`legislacao`** — Leis federais e seus artigos principais
- **`doutrina`** — Referências doutrinárias e acadêmicas

In [ ]:
import sqlite3
import os
import json
from datetime import datetime

# Cria diretório
os.makedirs(".", exist_ok=True)
DB_PATH = "juridico_segpub.db"

print(f"📦 Criando banco de dados: {DB_PATH}")
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Habilita WAL mode para performance
cursor.execute("PRAGMA journal_mode=WAL")

# ─────────────────────────────────────────────
# TABELA 1: ACÓRDÃOS
# ─────────────────────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS acordaos (
        id INTEGER PRIMARY KEY,
        numero TEXT NOT NULL UNIQUE,
        tribunal TEXT NOT NULL,
        data_julgamento TEXT,
        relator TEXT,
        tipo TEXT,
        ementa TEXT,
        resultado TEXT,
        pena_anos REAL,
        area_direito TEXT
    )
""")

# ─────────────────────────────────────────────
# TABELA 2: OCORRÊNCIAS
# ─────────────────────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS ocorrencias (
        id INTEGER PRIMARY KEY,
        numero_bo TEXT NOT NULL UNIQUE,
        data TEXT,
        tipo_crime TEXT,
        municipio TEXT,
        estado TEXT,
        status TEXT,
        valor_envolvido_reais REAL,
        descricao TEXT
    )
""")

# ─────────────────────────────────────────────
# TABELA 3: LEGISLAÇÃO
# ─────────────────────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS legislacao (
        id INTEGER PRIMARY KEY,
        numero TEXT NOT NULL UNIQUE,
        tipo TEXT,
        data_publicacao TEXT,
        ementa TEXT,
        artigos_principais TEXT,
        status_vigencia TEXT DEFAULT 'Vigente'
    )
""")

# ─────────────────────────────────────────────
# TABELA 4: DOUTRINA
# ─────────────────────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS doutrina (
        id INTEGER PRIMARY KEY,
        titulo TEXT,
        autores TEXT,
        publicacao TEXT,
        ano INTEGER,
        resumo TEXT,
        temas TEXT
    )
""")

conn.commit()
print("✅ Tabelas criadas")

In [ ]:
# INSERÇÃO DE ACÓRDÃOS

acordaos = [
    # Habeas Corpus — Prisão Preventiva
    (1, "HC 123456", "STF", "2023-03-15", "Min. Alexandre de Moraes", "Habeas Corpus",
     "Prisão preventiva em crime de corrupção passiva. Fundamentação concreta necessária. "
     "Critérios objetivos para decretação. Duração razoável do processo. "
     "Medidas cautelares alternativas como regra.",
     "Ordem concedida", None, "Direito Penal — Crimes contra Administração Pública"),
    
    (2, "HC 234567", "STF", "2023-06-20", "Min. Gilmar Mendes", "Habeas Corpus",
     "Prisão preventiva em crime de lavagem de dinheiro. Ausência de perigo concreto. "
     "Medidas cautelares suficientes para acautelar a instrução. Princípio da "
     "proporcionalidade. Paciente cooperando com as investigações.",
     "Ordem concedida parcialmente", None, "Direito Penal — Lavagem de Dinheiro"),
    
    (3, "HC 345678", "STJ", "2023-09-10", "Min. Rogério Schietti", "Habeas Corpus",
     "Tráfico de drogas. Prisão preventiva mantida. Garantia da ordem pública fundamentada "
     "em quantidade e qualidade da droga apreendida (50kg cocaína pureza 80%). "
     "Reiteração criminosa demonstrada.",
     "Ordem denegada", None, "Direito Penal — Tráfico de Drogas"),
    
    (4, "AP 987", "STF", "2023-11-05", "Min. Rosa Weber", "Ação Penal",
     "Crime de peculato praticado por parlamentar federal. Competência originária STF. "
     "Desvio de R$ 1,2 milhão de verbas de gabinete. Condenação com perda do mandato. "
     "Princípio da proporcionalidade na fixação da pena.",
     "Procedente — Condenação", 8.0, "Direito Penal — Crimes contra Administração Pública"),
    
    (5, "RHC 567890", "STJ", "2024-02-14", "Min. Laurita Vaz", "Recurso em Habeas Corpus",
     "Crime de estelionato via meios eletrônicos (golpe do PIX). Liberdade provisória. "
     "Fiança arbitrada. Condições pessoais favoráveis do réu. Ausência de reiteração.",
     "Provido", None, "Direito Penal — Crimes Patrimoniais"),
    
    (6, "RHC 150234", "STJ", "2023-08-22", "Min. Sebastião Reis Júnior", "Recurso em Habeas Corpus",
     "Lavagem de dinheiro — operação de grande porte. Prisão preventiva. "
     "Necessidade de fundamentação específica quanto à existência de perigo concreto ao "
     "processo. Mera gravidade abstrata do delito não basta.",
     "Provido", None, "Direito Penal — Lavagem de Dinheiro"),
    
    (7, "AP 996", "STF", "2023-12-01", "Min. Edson Fachin", "Ação Penal",
     "Lavagem de capitais com organização criminosa transnacional. "
     "Operação envolveu contas em paraísos fiscais e criptomoedas. "
     "Competência federal. Condenação à pena máxima considerando gravidade e capacidade econômica.",
     "Procedente — Condenação", 12.0, "Direito Penal — Lavagem de Dinheiro"),
    
    (8, "HC 567123", "STF", "2024-01-15", "Min. Alexandre de Moraes", "Habeas Corpus",
     "Lavagem de dinheiro. Bloqueio de ativos. Proporcionalidade. "
     "Bloqueio deve ser limitado ao valor do dano estimado, não de forma irrestrita. "
     "Direito de propriedade como limite constitucional.",
     "Ordem parcialmente concedida", None, "Direito Penal — Lavagem de Dinheiro"),
    
    (9, "AgRg RHC 180000", "STJ", "2024-03-20", "Min. Messod Azulay", "Agravo Regimental",
     "Improbidade administrativa. Indisponibilidade de bens. Fumus boni iuris e periculum "
     "in mora demonstrados. Manutenção da medida cautelar. Valor bloqueado proporcional "
     "ao dano estimado ao erário (R$ 3,5 milhões).",
     "Improvido — Medida mantida", None, "Direito Administrativo — Improbidade"),
    
    (10, "HC 901234", "STF", "2024-04-10", "Min. Cristiano Zanin", "Habeas Corpus",
     "Organização criminosa. Interceptação telefônica. Prazo de 15 dias. "
     "Renovações sucessivas. Limite de 60 dias consolidado na jurisprudência. "
     "Ilicitude das provas obtidas além do prazo.",
     "Ordem concedida", None, "Direito Processual Penal — Provas")
]

cursor.executemany(
    "INSERT OR REPLACE INTO acordaos VALUES (?,?,?,?,?,?,?,?,?,?)",
    acordaos
)
print(f"✅ {len(acordaos)} acórdãos inseridos")

In [ ]:
# INSERÇÃO DE OCORRÊNCIAS

ocorrencias = [
    (1, "BO-2024-001", "2024-01-15", "Corrupção Ativa", "São Paulo", "SP",
     "Em investigação", 150000.00,
     "Agente público solicitou vantagem indevida de R$ 150 mil para liberação de alvará de construção."),
    
    (2, "BO-2024-002", "2024-01-20", "Lavagem de Dinheiro", "Rio de Janeiro", "RJ",
     "Indiciado", 2300000.00,
     "COAF identificou movimentações suspeitas de R$ 2,3 milhões em 3 meses via contas laranja."),
    
    (3, "BO-2024-003", "2024-02-05", "Tráfico de Drogas", "Salvador", "BA",
     "Preso em flagrante", 0,
     "Apreensão de 50kg de cocaína com pureza de 80% em depósito clandestino em área portuária."),
    
    (4, "BO-2024-004", "2024-02-18", "Estelionato", "Curitiba", "PR",
     "Arquivado", 5000.00,
     "Golpe do PIX. Vítima transferiu R$ 5.000 para conta fraudulenta após contato via WhatsApp."),
    
    (5, "BO-2024-005", "2024-03-01", "Peculato", "Brasília", "DF",
     "Em investigação", 800000.00,
     "Desvio de verbas públicas de programa de saúde municipal. Suspeita envolve 3 servidores."),
    
    (6, "BO-2024-006", "2024-03-15", "Organização Criminosa", "Manaus", "AM",
     "Preso em flagrante", 5000000.00,
     "Desarticulação de ORCRIM com 8 membros especializados em fraude a licitações municipais."),
    
    (7, "BO-2024-007", "2024-03-22", "Improbidade Administrativa", "Porto Alegre", "RS",
     "Ação Civil Pública", 3500000.00,
     "Ex-secretário municipal desviou R$ 3,5 milhões de contratos de merenda escolar durante pandemia."),
    
    (8, "BO-2024-008", "2024-04-01", "Interceptação Ilegal", "Recife", "PE",
     "Inquérito policial", 0,
     "Denúncia de interceptação telefônica sem autorização judicial por agente de segurança privada."),
    
    (9, "BO-2024-009", "2024-04-10", "Corrupção Passiva", "Belo Horizonte", "MG",
     "Indiciado", 250000.00,
     "Fiscal tributário recebeu propina de R$ 250 mil para reduzir autuações de empresa."),
    
    (10, "BO-2024-010", "2024-01-15", "Lavagem de Dinheiro", "São Paulo", "SP",
     "Em investigação", 8500000.00,
     "COAF identificou movimentações suspeitas de R$ 8,5M em 3 meses via contas laranja "
     "ligadas a exchange de criptomoedas não regulamentada."),
    
    (11, "BO-2024-011", "2024-02-03", "Lavagem de Dinheiro", "Rio de Janeiro", "RJ",
     "Indiciado", 2300000.00,
     "Suspeito utilizou criptomoedas (Bitcoin e Monero) para dissimular origem de R$ 2,3M "
     "provenientes de tráfico de drogas. Rastreamento feito por análise de blockchain."),
    
    (12, "BO-2024-012", "2024-03-10", "Organização Criminosa", "Curitiba", "PR",
     "Preso em flagrante", 12000000.00,
     "Grupo criminoso com 12 membros operava esquema de lavagem de R$ 12M via "
     "imobiliárias de fachada. Cada imóvel era revendido para simular transações legítimas.")
]

cursor.executemany(
    "INSERT OR REPLACE INTO ocorrencias VALUES (?,?,?,?,?,?,?,?,?)",
    ocorrencias
)
print(f"✅ {len(ocorrencias)} ocorrências inseridas")

In [ ]:
# INSERÇÃO DE LEGISLAÇÃO

legislacao = [
    (1, "Lei 8.429/1992", "Lei Federal", "1992-06-02",
     "Lei de Improbidade Administrativa. Sanções aplicáveis a agentes públicos nos casos "
     "de enriquecimento ilícito, dano ao erário e violação de princípios administrativos. "
     "Alterada pela Lei 14.230/2021 — exige dolo específico para configuração do ato ímprobo.",
     "Art. 9 - Enriquecimento ilícito; Art. 10 - Dano ao erário; Art. 11 - Princípios; "
     "Art. 17 - Legitimidade para ação; Art. 23 - Prescrição",
     "Vigente"),
    
    (2, "Lei 9.296/1996", "Lei Federal", "1996-07-24",
     "Regulamenta interceptação de comunicações telefônicas e telemáticas para fins de "
     "investigação criminal. Requisitos: indício razoável, impossibilidade de outros meios, "
     "crime com pena de reclusão. Prazo: 15 dias, renovável por igual período.",
     "Art. 2 - Requisitos; Art. 3 - Requisição judicial; Art. 5 - Prazo 15 dias (renovável); "
     "Art. 10 - Crime de violação do sigilo",
     "Vigente"),
    
    (3, "Lei 9.613/1998", "Lei Federal", "1998-03-03",
     "Lei de Lavagem de Dinheiro. Crimes de lavagem ou ocultação de bens, direitos e valores. "
     "Prevenção do uso do sistema financeiro para ilícitos. Cria o COAF (Conselho de Controle "
     "de Atividades Financeiras). Inclui obrigações de identificação de clientes e reporte.",
     "Art. 1 - Crime de lavagem (pena 3-10 anos); Art. 9 - Obrigados a informar; "
     "Art. 11 - Comunicação de operações suspeitas; Art. 14 - Criação do COAF",
     "Vigente"),
    
    (4, "Lei 12.850/2013", "Lei Federal", "2013-08-02",
     "Lei das Organizações Criminosas. Define organização criminosa (4+ pessoas, estruturada, "
     "para prática de crimes com pena máxima >4 anos). Meios especiais de investigação: "
     "colaboração premiada, captação ambiental, infiltração de agentes, ação controlada.",
     "Art. 1 - Conceito de ORCRIM; Art. 3 - Meios de obtenção de prova; "
     "Art. 4 - Colaboração premiada; Art. 10 - Ação controlada; Art. 11 - Infiltração",
     "Vigente"),
    
    (5, "Lei 14.230/2021", "Lei Federal", "2021-10-25",
     "Altera substancialmente a Lei de Improbidade Administrativa (8.429/92). "
     "Exige dolo específico para configuração de ato de improbidade. Extingue modalidade culposa. "
     "Prazo prescricional: 8 anos. Retroatividade in bonam partem.",
     "Art. 1 - Dolo específico exigido; Art. 3 - Sujeito ativo; "
     "Art. 23 - Prescrição 8 anos; Art. 1º, §4º - Retroatividade",
     "Vigente"),
    
    (6, "Decreto-Lei 2.848/1940", "Código Penal", "1940-12-07",
     "Código Penal Brasileiro. Parte Especial — Crimes contra a Administração Pública: "
     "peculato (Art. 312), concussão (Art. 316), corrupção passiva (Art. 317), "
     "corrupção ativa (Art. 333).",
     "Art. 312 - Peculato (2-12 anos); Art. 316 - Concussão (2-8 anos); "
     "Art. 317 - Corrupção passiva (2-12 anos); Art. 333 - Corrupção ativa (2-12 anos)",
     "Vigente")
]

cursor.executemany(
    "INSERT OR REPLACE INTO legislacao VALUES (?,?,?,?,?,?,?)",
    legislacao
)
print(f"✅ {len(legislacao)} leis inseridas")

In [ ]:
# INSERÇÃO DE DOUTRINA (referências acadêmicas)

doutrina_refs = [
    (1, "ReAct: Synergizing Reasoning and Acting in Language Models",
     "Yao, Shunyu; Zhao, Jeffrey; Yu, Dian; Du, Nan et al.",
     "ICLR 2023", 2023,
     "Propõe o paradigma ReAct que combina raciocínio (Thought) e ação (Action) em loops "
     "para LLMs. Demonstra superioridade sobre Chain-of-Thought em tarefas de QA e "
     "tomada de decisão interativa. Base do Agentic RAG.",
     "ReAct, Agentes LLM, Raciocínio, Tool Use"),
    
    (2, "Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity",
     "Jeong, Soyeong; Baek, Jinheon; Cho, Sukmin et al.",
     "ACL Findings 2024", 2024,
     "Propõe classificador de complexidade para rotear queries para estratégias RAG "
     "distintas. Três caminhos: sem retrieval, single-step, multi-step. "
     "Otimiza custo, latência e qualidade em sistemas de produção.",
     "Adaptive RAG, Classificador, Complexidade, Produção"),
    
    (3, "LangGraph: Build Stateful, Multi-Actor Applications with LLMs",
     "LangChain-AI Team",
     "Documentação Oficial — langchain-ai.github.io", 2024,
     "Framework para construção de agentes LLM como grafos de estado. "
     "Suporta ciclos, memória persistente e múltiplos agentes. "
     "Baseado no conceito de StateGraph com nós e arestas condicionais.",
     "LangGraph, Agentes, StateGraph, Memória")
]

cursor.executemany(
    "INSERT OR REPLACE INTO doutrina VALUES (?,?,?,?,?,?,?)",
    doutrina_refs
)
conn.commit()
conn.close()
print(f"✅ {len(doutrina_refs)} referências doutrinárias inseridas")

In [ ]:
# VALIDAÇÃO DO DATASET

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("📊 SUMÁRIO DO DATASET")
print("=" * 50)

tabelas = ["acordaos", "ocorrencias", "legislacao", "doutrina"]
for tabela in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabela}")
    n = cursor.fetchone()[0]
    print(f"  📋 {tabela:<20}: {n:>3} registros")

print("\n📊 ACÓRDÃOS POR TRIBUNAL:")
cursor.execute("SELECT tribunal, COUNT(*) as n FROM acordaos GROUP BY tribunal ORDER BY n DESC")
for tribunal, n in cursor.fetchall():
    print(f"  {tribunal}: {n}")

print("\n📊 OCORRÊNCIAS POR TIPO DE CRIME:")
cursor.execute("SELECT tipo_crime, COUNT(*) as n FROM ocorrencias GROUP BY tipo_crime ORDER BY n DESC")
for tipo, n in cursor.fetchall():
    print(f"  {tipo}: {n}")

print("\n📊 VALOR TOTAL POR ESTADO (Ocorrências):")
cursor.execute("""
    SELECT estado, COUNT(*) as n, SUM(valor_envolvido_reais) as total_reais
    FROM ocorrencias 
    WHERE valor_envolvido_reais > 0
    GROUP BY estado
    ORDER BY total_reais DESC
""")
for estado, n, total in cursor.fetchall():
    print(f"  {estado}: {n} casos | R$ {total:,.0f}")

conn.close()
print(f"\n✅ Dataset validado e pronto para uso!")
print(f"   Arquivo: {DB_PATH}")

# Exporta sumário como JSON para referência
sumario = {
    "arquivo": DB_PATH,
    "criado_em": datetime.now().isoformat(),
    "tabelas": tabelas,
    "uso": "Labs 1, 2 e 3 da Aula 10 — Agentic RAG e Adaptive RAG"
}
with open("dataset_sumario.json", "w") as f:
    json.dump(sumario, f, indent=2, ensure_ascii=False)
print(f"   Sumário: dataset_sumario.json")